In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class MobileNetV3SmallReviewKDWrapper(nn.Module):
  """Unrolled, layer-by-layer trainable wrapper for MobileNetV3-Small (width_mult=0.5).

  Extracts ReviewKD features at resolution changes without block grouping.
  """

  def __init__(self, model: nn.Module):
    super().__init__()
    # Keep the original submodules registered so all parameters train normally
    self.features = model.features
    self.classifier = model.classifier

    # Infer channel counts dynamically for ReviewKD configs
    self.stage_channels = self._get_channels()

  def _get_channels(self):
    self.eval()
    with torch.no_grad():
      _, feats = self.forward(torch.zeros(1, 3, 224, 224))
      channels = [f.shape[1] for f in feats["preact_feats"]]
    self.train()
    return channels

  def get_bn_before_relu(self):
    """Directly returns the trailing BatchNorm layer for each stage."""
    return [
        self.features[1].block[-1][1],  # Stage 1 terminal BN (56x56)
        self.features[3].block[-1][1],  # Stage 2 terminal BN (28x28)
        self.features[8].block[-1][1],  # Stage 3 terminal BN (14x14)
        self.features[12][1],  # Stage 4 terminal BN (7x7)
    ]

  def get_stage_channels(self):
    return self.stage_channels

  def forward(self, x):
    # --- Stem (224x224 -> 112x112) ---
    x = self.features[0](x)

    # --- Stage 1 (112x112 -> 56x56) ---
    x = self.features[1](x)
    f_56 = x  # Tap Level 1: (B, C1, 56, 56)

    # --- Stage 2 (56x56 -> 28x28) ---
    x = self.features[2](x)  # stride=2 downsample
    x = self.features[3](x)
    f_28 = x  # Tap Level 2: (B, C2, 28, 28)

    # --- Stage 3 (28x28 -> 14x14) ---
    x = self.features[4](x)  # stride=2 downsample
    x = self.features[5](x)
    x = self.features[6](x)
    x = self.features[7](x)
    x = self.features[8](x)
    f_14 = x  # Tap Level 3: (B, C3, 14, 14)

    # --- Stage 4 (14x14 -> 7x7) ---
    x = self.features[9](x)  # stride=2 downsample
    x = self.features[10](x)
    x = self.features[11](x)

    # Final expansion block: features[12] = Conv2d(0) -> BatchNorm(1) -> Hardswish(2)
    x = self.features[12][0](x)  # 1x1 Conv
    f_7 = self.features[12][1](x)  # Tap Level 4: Post-BN, Pre-Hardswish (7x7)
    x = self.features[12][2](f_7)  # Hardswish

    # --- Stage 5 (Global Representation: 1x1) ---
    avg_pool = F.adaptive_avg_pool2d(x, (1, 1))
    pooled = avg_pool.flatten(1) # Flatten to (B, C4) for classifier input

    # Standard classification head
    out = self.classifier(pooled)

    feats = {
        # Ordered apex-to-base: [f_56, f_28, f_14, f_7]
        "preact_feats": [f_56, f_28, f_14, f_7],
        "pooled_feat": pooled,
    }

    return out, feats

In [9]:
class DenseNet201(nn.Module):
  """Wraps a pre-trained torchvision DenseNet model to extract both raw

  DenseBlock outputs and pre-ReLU transition features for ReviewKD.
  preact_feats: [stem, f1_pre, f2_pre, f3_pre, f4_pre] representing the pre-ReLU features from the stem and each of the four DenseBlocks.
  earlier_feats: [earlier_feat1, earlier_feat2, earlier_feat3, earlier_feat4] representing eariler feature maps of DenseNet's stage.
  """

  def __init__(self, model: nn.Module = models.densenet201(weights=None)):
    super().__init__()
    self.model = model
    self.features = model.features
    self.classifier = model.classifier

    # Output channels for DenseNet-201: [56x56, 28x28, 14x14, 7x7, 1x1]
    self.stage_channels = [256, 512, 1792, 1920, 1920]

  def get_bn_before_relu(self):
    return [
        self.features.transition1.norm,
        self.features.transition2.norm,
        self.features.transition3.norm,
        self.features.norm5,
    ]

  def get_stage_channels(self):
    return self.stage_channels

  def forward(self, x):
    # Stem: 224x224 -> 56x56
    x = self.features.conv0(x) # input size from 3x224x224 to 64x112x112
    x = self.features.norm0(x) # input size from 64x112x112 to 64x112x112
    stem = x
    x = self.features.relu0(x) # input size from 64x112x112 to 64x112x112
    x = self.features.pool0(x) # input size from 64x112x112 to 64x56x56
    earlier_feat1 = x

    # Stage 1 (56x56)
    db1_out = self.features.denseblock1(x)  # Raw DenseBlock 1
    f1_pre = self.features.transition1.norm(
        db1_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition1.pool(
        self.features.transition1.conv(self.features.transition1.relu(f1_pre))
    )
    earlier_feat2 = x

    # Stage 2 (28x28)
    db2_out = self.features.denseblock2(x)  # Raw DenseBlock 2
    f2_pre = self.features.transition2.norm(
        db2_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition2.pool(
        self.features.transition2.conv(self.features.transition2.relu(f2_pre))
    )
    earlier_feat3 = x

    # Stage 3 (14x14)
    db3_out = self.features.denseblock3(x)  # Raw DenseBlock 3
    f3_pre = self.features.transition3.norm(
        db3_out
    )  # Pre-ReLU normalized feature
    x = self.features.transition3.pool(
        self.features.transition3.conv(self.features.transition3.relu(f3_pre))
    )
    earlier_feat4 = x

    # Stage 4 (7x7)
    db4_out = self.features.denseblock4(x)  # Raw DenseBlock 4
    f4_pre = self.features.norm5(db4_out)  # Pre-ReLU normalized feature
    f4 = F.relu(f4_pre, inplace=True)

    # Stage 5 (1x1 GAP)
    f5_gap = F.adaptive_avg_pool2d(f4, (1, 1))  # (B, 1920, 1, 1)
    pooled = f5_gap.flatten(1) # shape: from (B, 1920, 1, 1) to (B, 1920)
    out = self.classifier(pooled)

    feats = {
        "denseblock_feats": [db1_out, db2_out, db3_out, db4_out],
        "earlier_feats": [earlier_feat1, earlier_feat2, earlier_feat3, earlier_feat4],
        "preact_feats": [stem, f1_pre, f2_pre, f3_pre, f4_pre],
        "pooled_feat": pooled,
    }

    return out, feats

In [10]:
# create mobilenetv3_small 0.5 without pretrained weights
mobilenetv3_model = models.mobilenet_v3_small(weights=None)
mobilenetv3_student = MobileNetV3SmallReviewKDWrapper(mobilenetv3_model)

In [11]:
densenet201_model = models.densenet201(weights=None)
densenet201_teacers = DenseNet201(densenet201_model)

In [12]:
mobilenetv3_model = models.mobilenet_v3_small(weights=None)

In [13]:
mobilenetv3_model

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
# ---------------- Activations ----------------
class HardSigmoid(nn.Module):
    def forward(self, input_tensor):
        return F.relu6(input_tensor + 3.0, inplace=True) / 6.0


class HardSwish(nn.Module):
    def forward(self, input_tensor):
        return input_tensor * F.relu6(input_tensor + 3.0, inplace=True) / 6.0

# ---------------- Basic layers ----------------
def conv_batchnorm_act(
    input_channels: int,
    output_channels: int,
    kernel_size: int = 3,
    stride: int = 1,
    groups: int = 1,
    activation_type: str = "relu",
):
    padding = (kernel_size - 1) // 2

    if activation_type == "relu":
        activation_layer = nn.ReLU(inplace=True)
    elif activation_type == "hard_swish":
        activation_layer = HardSwish()
    else:
        raise ValueError("Unsupported activation")

    return nn.Sequential(
        nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=groups,
            bias=False,
        ),
        nn.BatchNorm2d(output_channels),
        activation_layer,
    )

class SqueezeExcitation(nn.Module):
    def __init__(
        self,
        input_channels: int,
        squeeze_ratio: float = 0.25,
    ):
        super().__init__()
        squeezed_channels = max(8, int(input_channels * squeeze_ratio))

        self.squeeze_excitation = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(input_channels, squeezed_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(squeezed_channels, input_channels, kernel_size=1),
            HardSigmoid(),
        )

    def forward(self, input_tensor):
        return input_tensor * self.squeeze_excitation(input_tensor)

class InvertedResidualBlock(nn.Module):
    def __init__(
        self,
        input_channels: int,
        expanded_channels: int,
        output_channels: int,
        depthwise_kernel_size: int,
        stride: int,
        use_squeeze_excitation: bool,
        activation_type: str,
    ):
        super().__init__()

        self.use_residual_connection = (
            stride == 1 and input_channels == output_channels
        )

        self.expansion_layer = (
            conv_batchnorm_act(
                input_channels,
                expanded_channels,
                kernel_size=1,
                stride=1,
                activation_type=activation_type,
            )
            if input_channels != expanded_channels
            else nn.Identity()
        )

        self.depthwise_convolution = conv_batchnorm_act(
            expanded_channels,
            expanded_channels,
            kernel_size=depthwise_kernel_size,
            stride=stride,
            groups=expanded_channels,
            activation_type=activation_type,
        )

        self.squeeze_excitation = (
            SqueezeExcitation(expanded_channels)
            if use_squeeze_excitation
            else nn.Identity()
        )
        self.projection_layer = nn.Sequential(
            nn.Conv2d(
                expanded_channels,
                output_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(output_channels),
        )

    def forward(self, input_tensor):
        output = self.expansion_layer(input_tensor)
        output = self.depthwise_convolution(output)
        output = self.squeeze_excitation(output)
        output = self.projection_layer(output)

        if self.use_residual_connection:
            return input_tensor + output
        return output

In [44]:
class MobileNetV3Small(nn.Module):
    def __init__(
        self,
        num_classes: int = 1000,
        dropout_probability: float = 0.2,
    ):
        super().__init__()

        self.stem_layer = conv_batchnorm_act(
            input_channels=3,
            output_channels=16,
            kernel_size=3,
            stride=2,
            activation_type="hard_swish",
        )

        # in_ channels, expanded channels, out channels, depthwise kernel size, stride, use_se, activation

        self.block_1  = InvertedResidualBlock(16, 16, 16, 3, 2, True,  "relu")
        self.block_2  = InvertedResidualBlock(16, 72, 24, 3, 2, False, "relu")
        self.block_3  = InvertedResidualBlock(24, 88, 24, 3, 1, False, "relu")

        self.block_4  = InvertedResidualBlock(24, 96, 40, 5, 2, True,  "hard_swish")
        self.block_5  = InvertedResidualBlock(40,240, 40, 5, 1, True,  "hard_swish")
        self.block_6  = InvertedResidualBlock(40,240, 40, 5, 1, True,  "hard_swish")

        self.block_7  = InvertedResidualBlock(40,120, 48, 5, 1, True,  "hard_swish")
        self.block_8  = InvertedResidualBlock(48,144, 48, 5, 1, True,  "hard_swish")

        self.block_9  = InvertedResidualBlock(48,288, 96, 5, 2, True,  "hard_swish")
        self.block_10 = InvertedResidualBlock(96,576, 96, 5, 1, True,  "hard_swish")
        self.block_11 = InvertedResidualBlock(96,576, 96, 5, 1, True,  "hard_swish")

        self.final_conv = conv_batchnorm_act(
            input_channels=96,
            output_channels=576,
            kernel_size=1,
            activation_type="hard_swish",
        )

        self.global_average_pooling = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Conv2d(576, 1024, kernel_size=1, bias=True),
            HardSwish(),
            nn.Dropout(p=dropout_probability),
            nn.Conv2d(1024, num_classes, kernel_size=1, bias=True),
        )

    def forward(self, input_tensor):
        x = self.stem_layer(input_tensor)

        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = self.block_4(x)
        x = self.block_5(x)
        x = self.block_6(x)
        x = self.block_7(x)
        x = self.block_8(x)
        x = self.block_9(x)
        x = self.block_10(x)
        x = self.block_11(x)

        x = self.final_conv(x)
        x = self.global_average_pooling(x)
        x = self.classifier(x)
        return x.flatten(1)

In [48]:
import torch
import torch.nn as nn

class MobileNetV3(nn.Module):
    def __init__(
        self,
        blocks: list,
        last_block_out_channels: int,
        final_conv_channels: int,
        classifier_channels: int,
        num_classes: int = 1000,
        dropout_probability: float = 0.2,
    ):
        super().__init__()

        self.stem_layer = conv_batchnorm_act(
            input_channels=3,
            output_channels=16,
            kernel_size=3,
            stride=2,
            activation_type="hard_swish",
        )

        # Expand the blocks list directly into nn.Sequential (No loops)
        self.blocks = nn.Sequential(*blocks)

        self.final_conv = conv_batchnorm_act(
            input_channels=last_block_out_channels,
            output_channels=final_conv_channels,
            kernel_size=1,
            activation_type="hard_swish",
        )

        self.global_average_pooling = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Conv2d(final_conv_channels, classifier_channels, kernel_size=1, bias=True),
            HardSwish(),
            nn.Dropout(p=dropout_probability),
            nn.Conv2d(classifier_channels, num_classes, kernel_size=1, bias=True),
        )

    def forward(self, input_tensor):
        x = self.stem_layer(input_tensor)
        x = self.blocks(x)
        x = self.final_conv(x)
        x = self.global_average_pooling(x)
        x = self.classifier(x)
        return x.flatten(1)


# ---------------- Model Builders (Explicitly Expanded) ----------------

def mobilenet_v3_small(num_classes: int = 1000, dropout_probability: float = 0.2) -> MobileNetV3:
    """Builds the MobileNetV3-Small architecture by explicitly defining the 11 blocks."""
    
    # in_channels, expanded_channels, out_channels, depthwise_kernel, stride, use_se, activation
    blocks = [
        InvertedResidualBlock(16, 16, 16, 3, 2, True, "relu"),
        InvertedResidualBlock(16, 72, 24, 3, 2, False, "relu"),
        InvertedResidualBlock(24, 88, 24, 3, 1, False, "relu"),
        InvertedResidualBlock(24, 96, 40, 5, 2, True, "hard_swish"),
        InvertedResidualBlock(40, 240, 40, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(40, 240, 40, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(40, 120, 48, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(48, 144, 48, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(48, 288, 96, 5, 2, True, "hard_swish"),
        InvertedResidualBlock(96, 576, 96, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(96, 576, 96, 5, 1, True, "hard_swish"),
    ]
    
    return MobileNetV3(
        blocks=blocks,
        last_block_out_channels=96,
        final_conv_channels=576,
        classifier_channels=1024,
        num_classes=num_classes,
        dropout_probability=dropout_probability,
    )

def mobilenet_v3_large(num_classes: int = 1000, dropout_probability: float = 0.2) -> MobileNetV3:
    """Builds the MobileNetV3-Large architecture by explicitly defining the 15 blocks."""
    
    # in_channels, expanded_channels, out_channels, depthwise_kernel, stride, use_se, activation
    blocks = [
        InvertedResidualBlock(16, 16, 16, 3, 1, False, "relu"),
        InvertedResidualBlock(16, 64, 24, 3, 2, False, "relu"),
        InvertedResidualBlock(24, 72, 24, 3, 1, False, "relu"),
        InvertedResidualBlock(24, 72, 40, 5, 2, True, "relu"),
        InvertedResidualBlock(40, 120, 40, 5, 1, True, "relu"),
        InvertedResidualBlock(40, 120, 40, 5, 1, True, "relu"),
        InvertedResidualBlock(40, 240, 80, 3, 2, False, "hard_swish"),
        InvertedResidualBlock(80, 200, 80, 3, 1, False, "hard_swish"),
        InvertedResidualBlock(80, 184, 80, 3, 1, False, "hard_swish"),
        InvertedResidualBlock(80, 184, 80, 3, 1, False, "hard_swish"),
        InvertedResidualBlock(80, 480, 112, 3, 1, True, "hard_swish"),
        InvertedResidualBlock(112, 672, 112, 3, 1, True, "hard_swish"),
        InvertedResidualBlock(112, 672, 160, 5, 2, True, "hard_swish"),
        InvertedResidualBlock(160, 960, 160, 5, 1, True, "hard_swish"),
        InvertedResidualBlock(160, 960, 160, 5, 1, True, "hard_swish"),
    ]
    
    return MobileNetV3(
        blocks=blocks,
        last_block_out_channels=160,
        final_conv_channels=960,
        classifier_channels=1280,
        num_classes=num_classes,
        dropout_probability=dropout_probability,
    )

In [45]:
# create mobilenetv3_small 0.5 without pretrained weights
mobilenetv3_model = models.mobilenet_v3_small(weights=None, num_classes=10)

# count parameters
total_params = sum(p.numel() for p in mobilenetv3_model.parameters())
print(f"Total parameters in MobileNetV3-Small (width_mult=0.3): {total_params}")

Total parameters in MobileNetV3-Small (width_mult=0.3): 1528106


In [47]:
# create mobilenetv3_small 0.5 without pretrained weights
mobilenetv3_model = MobileNetV3Small(num_classes=10, dropout_probability=0)

# count parameters
total_params = sum(p.numel() for p in mobilenetv3_model.parameters())
print(f"Total parameters in MobileNetV3-Small (width_mult=0.3): {total_params}")

Total parameters in MobileNetV3-Small (width_mult=0.3): 1522620
